# LECTURE 2: DATA TRANSFORMATION (WEEK 4)
---
## Scaling, encoding, distribution transforms, feature expansion, and target transformation

In this lecture, we will explore how to convert raw data into a well-structured representation that improves learning, evaluation, and deployment. Most algorithms do not work well with unprocessed raw data. Models require numerical inputs rather than text labels. Distance-based models (like SVM and KNN) and linear models are sensitive to feature scale. Some models also benefit when distributions are closer to Gaussian.

# Part I: Feature Transformation
---
We start by understanding our variable types:
- **Numerical**: Integer (transactions), Float (return, volatility).
- **Categorical**: Nominal (sector, country), Ordinal (credit grade), Boolean (default flag).

## 1.1 Scaling Numerical Data: MinMaxScaler

### 1. Overview
Features measured in different units can distort distances, gradients, and regularization penalties. MinMaxScaler maps values to a specified range, usually $[0,1]$.

### 2. Concept & Formula
The formula for MinMaxScaler is:
$$ x' = \frac{x - x_{min}}{x_{max} - x_{min}} $$
**When to use**: Distance-based methods like KNN, bounded inputs, or situations where a fixed output range is meaningful.

### 3. Simple Explanation
Imagine grading students on a scale of 0 to 100, but the actual test was out of 50. To make it out of 100, you stretch the scores so the minimum score becomes 0 and the maximum becomes 100. Similarly, MinMaxScaler stretches or squeezes all values so they fit perfectly into a 0 to 1 box.

### 4. Manual Calculation
**Toy Income Data**: For training values $x = (20, 30, 50)$ (in USD thousands).
1. $x_{min} = 20$, $x_{max} = 50$.
2. For $x = 20$: $x' = (20 - 20) / (50 - 20) = 0$.
3. For $x = 30$: $x' = (30 - 20) / (50 - 20) = 10 / 30 = 0.333$.
4. For $x = 50$: $x' = (50 - 20) / (50 - 20) = 30 / 30 = 1$.

**What about new data?**
If a new test value 60 appears, it maps to: $(60 - 20) / 30 = 40/30 = 1.333$.
Notice how it exceeds the 0-1 range. You can use `clip=True` if you strictly want to cap it.

### 5. Code Python


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

train = np.array([20, 30, 50]).reshape(-1, 1)
test = np.array([60]).reshape(-1, 1)

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(train)
X_test_scaled = scaler.transform(test)

print("Data Min:", scaler.data_min_)
print("Data Max:", scaler.data_max_)
print("Train Scaled:", X_train_scaled.ravel())
print("Test Scaled:", X_test_scaled.ravel())


### 6. Finance Perspective
MinMaxScaler is often used in deep learning for finance, such as normalizing image-like financial tensors (e.g., limit order books) where a strictly bounded 0-1 input prevents vanishing or exploding gradients. However, a major drawback is its extreme sensitivity to outliers. One massive trade can compress 99% of normal trades into a tiny range near 0.

---

## 1.2 Scaling Numerical Data: StandardScaler

### 1. Overview
StandardScaler centers a feature at zero and scales it to unit standard deviation. It is a strong default for many numerical workflows, especially linear regression, logistic regression, SVM, and PCA.

### 2. Concept & Formula
$$ z = \frac{x - \mu}{\sigma} $$
Where $\mu$ is the mean and $\sigma$ is the standard deviation of the feature.

### 3. Simple Explanation
Instead of trapping data in a box like MinMaxScaler, StandardScaler translates your data so that the average is exactly 0. Then it scales the data so that 1 unit means "1 standard deviation away from the average." It tells you exactly how "abnormal" a data point is compared to the group.

### 4. Manual Calculation
**Toy Data**: $x = (2, 4, 6)$
1. Mean $\mu = (2 + 4 + 6) / 3 = 4$.
2. Variance = $[(2-4)^2 + (4-4)^2 + (6-4)^2] / 3 = [4 + 0 + 4] / 3 = 8/3$.
3. Standard Deviation $\sigma = \sqrt{8/3} \approx 1.633$.
4. For $x = 2$: $z = (2 - 4) / 1.633 \approx -1.225$.
5. For $x = 4$: $z = (4 - 4) / 1.633 = 0$.
6. For $x = 6$: $z = (6 - 4) / 1.633 \approx 1.225$.

**What about new data?**
A new value 8 maps to $(8 - 4) / 1.633 \approx 2.45$ using training statistics.
*Question: Does StandardScaler make a skewed distribution normal?* **No.** It just shifts and scales.

### 5. Code Python


In [ ]:
from sklearn.preprocessing import StandardScaler

train = np.array([2, 4, 6]).reshape(-1, 1)
test = np.array([8]).reshape(-1, 1)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(train)
X_test_scaled = scaler.transform(test)

print("Mean:", scaler.mean_)
print("Scale (Std Dev):", scaler.scale_)
print("Train Scaled:", X_train_scaled.ravel())
print("Test Scaled:", X_test_scaled.ravel())


### 6. Finance Perspective
Standardizing asset returns into Z-scores is a fundamental step in quantitative finance for cross-sectional momentum strategies. By scaling returns of different stocks to $N(0,1)$, a quant can compare the relative performance of a highly volatile tech stock against a stable utility stock fairly.

---

## 1.3 Scaling Numerical Data: RobustScaler

### 1. Overview
RobustScaler uses robust statistics (median and Interquartile Range - IQR) rather than mean and standard deviation. It is highly effective when genuine extreme values are common.

### 2. Concept & Formula
$$ x' = \frac{x - Q_2}{Q_3 - Q_1} = \frac{x - \text{median}(X)}{\text{IQR}} $$

### 3. Simple Explanation
StandardScaler is easily ruined by a single crazy outlier because the outlier drags the mean and standard deviation with it. RobustScaler ignores the extremes and only looks at the middle 50% of the data to decide how to scale.

### 4. Manual Calculation
**Data with an outlier**: $(10, 12, 14, 16, 100)$
1. $Q_1 = 12$, $Q_2 (\text{median}) = 14$, $Q_3 = 16$.
2. $\text{IQR} = 16 - 12 = 4$.
3. For $x=10$: $x' = (10 - 14) / 4 = -1$.
4. For $x=16$: $x' = (16 - 14) / 4 = 0.5$.
5. For $x=100$: $x' = (100 - 14) / 4 = 21.5$.

Notice that the outlier (100) stays an outlier (21.5), but the normal values are beautifully scaled without being compressed to zero like in MinMaxScaler.

### 5. Code Python


In [ ]:
from sklearn.preprocessing import RobustScaler

train = np.array([10, 12, 14, 16, 100]).reshape(-1, 1)

scaler = RobustScaler()
X_scaled = scaler.fit_transform(train)

print('Median (Center):', scaler.center_)
print('IQR (Scale):', scaler.scale_)
print("Scaled Data:", X_scaled.ravel())


### 6. Finance Perspective
RobustScaler is critical when working with corporate fundamentals, such as P/E ratios. A few companies might have P/E ratios of 10,000 due to near-zero earnings. Using StandardScaler would squish all normal P/E ratios (e.g., 15-25) into a tiny band. RobustScaler keeps the normal companies properly spaced out.

---

## 1.4 Encoding Categorical Data: OrdinalEncoder & OneHotEncoder

### 1. Overview
Models require numerical inputs.
- **Ordinal categories** have a meaningful order (Small < Medium < Large).
- **Nominal categories** have no intrinsic ranking (Country, Sector).

### 2. Concept & Formula
- **OrdinalEncoder**: Maps ordered categories to integers (e.g., $S \mapsto 0$, $M \mapsto 1$, $L \mapsto 2$).
- **OneHotEncoder**: Creates one binary column per nominal category (e.g., Red $\mapsto (1,0,0)$, Green $\mapsto (0,1,0)$).

### 3. Simple Explanation
If you use OrdinalEncoder for nominal data (like Countries: $VN \mapsto 1$, $US \mapsto 2$, $UK \mapsto 3$), the model will mathematically think that $UK > US > VN$ and that the average of $VN$ and $UK$ is $US$. This is absurd. That's why we use OneHotEncoder for nominal data, giving each category its own independent Yes/No column.

### 4. Manual Calculation
**Toy Customer Dataset**:
- C1: Risk = Low, Region = North
- C2: Risk = Medium, Region = South
- C3: Risk = High, Region = North
- C4: Risk = Medium, Region = Central

**Ordinal for Risk**: Low $\mapsto 0$, Medium $\mapsto 1$, High $\mapsto 2$. Result: $(0, 1, 2, 1)$.
**One-Hot for Region** (Columns: Central, North, South):
- North: $(0, 1, 0)$
- South: $(0, 0, 1)$
- North: $(0, 1, 0)$
- Central: $(1, 0, 0)$

### 5. Code Python


In [ ]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

df_customers = pd.DataFrame({
    'customer': ['C1', 'C2', 'C3', 'C4'],
    'risk': ['Low', 'Medium', 'High', 'Medium'],
    'region': ['North', 'South', 'North', 'Central']
})

# Ordinal Encoding for Risk
risk_order = [['Low', 'Medium', 'High']]
ord_enc = OrdinalEncoder(categories=risk_order)
df_customers['risk_encoded'] = ord_enc.fit_transform(df_customers[['risk']])

# One-Hot Encoding for Region
ohe = OneHotEncoder(sparse_output=False)
region_encoded = ohe.fit_transform(df_customers[['region']])
region_cols = ohe.get_feature_names_out(['region'])
df_region = pd.DataFrame(region_encoded, columns=region_cols)

print("Encoded Data:")
print(pd.concat([df_customers, df_region], axis=1))


### 6. Finance Perspective
One-hot encoding is heavily used for categorical features like "Industry Sector" or "Exchange Name". However, beware of the "Dummy Variable Trap" (multicollinearity) in linear regression, where you might need to drop one category (using `drop='first'`). Also, high cardinality (e.g., ZIP codes) can explode your feature space, requiring advanced embeddings or Target Encoding instead.

---

# Part II: Distribution and Other Transforms
---

## 2.1 PowerTransformer (Box-Cox and Yeo-Johnson)

### 1. Overview
Power transformers alter the shape of a distribution to reduce strong skewness, stabilize variance, and make the data more Gaussian-like. 

### 2. Concept & Formula
**Box-Cox** (for strictly positive data $y > 0$):
$$ y^{(\lambda)} = \frac{y^\lambda - 1}{\lambda} \quad (\text{if } \lambda \neq 0) $$
**Yeo-Johnson**: Extends Box-Cox to handle positive, zero, and negative values. The parameter $\lambda$ is automatically estimated from the data via maximum likelihood.

### 3. Simple Explanation
Think of taking the square root or logarithm of a number. A huge difference like $10,000$ vs $1,000,000$ shrinks down when you take the log ($4$ vs $6$). Power transformers automatically find the best "power" ($\lambda$) to compress long tails so that the data looks like a nice, symmetric bell curve.

### 4. Manual Calculation
**Toy Claim Amounts**: Right-skewed positive claims $y = (1, 4, 9, 25, 100)$.
Let's manually use Box-Cox with $\lambda = 0.5$:
$$ y^{(0.5)} = \frac{y^{0.5} - 1}{0.5} = 2(\sqrt{y} - 1) $$
- For 1: $2(1 - 1) = 0$
- For 4: $2(2 - 1) = 2$
- For 9: $2(3 - 1) = 4$
- For 25: $2(5 - 1) = 8$
- For 100: $2(10 - 1) = 18$

The original gap between the last two is $100 - 25 = 75$. The new gap is $18 - 8 = 10$. The long tail is compressed!

### 5. Code Python


In [ ]:
from sklearn.preprocessing import PowerTransformer

skewed_data = np.array([1, 4, 9, 25, 100]).reshape(-1, 1)

# We use Yeo-Johnson as it is the flexible default in scikit-learn
# Note: standardize=True (default) will also Z-score scale the output
pt = PowerTransformer(method='yeo-johnson', standardize=False)
transformed_data = pt.fit_transform(skewed_data)

print("Estimated Lambda:", pt.lambdas_)
print("Transformed Data:\n", transformed_data.ravel())


### 6. Finance Perspective
Many financial variables (like trade volume, AUM, market cap, insurance claims) are extremely right-skewed (log-normal distribution). Applying a PowerTransformer helps linear models (which assume Gaussian errors) fit the data much better without being completely derailed by mega-cap companies or whale trades.

---

## 2.2 QuantileTransformer

### 1. Overview
QuantileTransformer maps empirical ranks to a uniform (between 0 and 1) or normal distribution. It is highly robust to outliers because it only cares about the *rank* (position) of the data, not its actual magnitude.

### 2. Concept & Formula
For a uniform output, it assigns positions:
$$ u_i = \frac{\text{rank}(x_i) - 1}{N - 1} $$

### 3. Simple Explanation
Imagine a race where runners finish at 10 mins, 20 mins, 30 mins, and 100 mins. Instead of caring about the time gap, we just say: 1st place, 2nd place, 3rd place, 4th place. We map these places evenly from 0% to 100%. The massive time gap of the last runner is completely ignored.

### 4. Manual Calculation
**Toy Transaction Values**: $x = (10, 20, 30, 100)$. $N = 4$.
- 10 is rank 1: $u = (1-1)/3 = 0$
- 20 is rank 2: $u = (2-1)/3 = 1/3 \approx 0.333$
- 30 is rank 3: $u = (3-1)/3 = 2/3 \approx 0.667$
- 100 is rank 4: $u = (4-1)/3 = 1$

Notice how $(10, 20, 30, 100) \mapsto (0, 0.33, 0.66, 1)$. The gap is perfectly equalized.

### 5. Code Python


In [ ]:
from sklearn.preprocessing import QuantileTransformer

data = np.array([10, 20, 30, 100]).reshape(-1, 1)

qt_uniform = QuantileTransformer(output_distribution='uniform', n_quantiles=4)
uniform_data = qt_uniform.fit_transform(data)

print("Uniform Quantile Data:\n", uniform_data.ravel())


### 6. Finance Perspective
Quantile transforms are the secret weapon for cross-sectional ranking models. If you rank 500 stocks based on their Momentum signal and map them to a uniform distribution, you neutralize extreme market shocks (like a stock going up 500% in a day). Your model learns purely from the relative hierarchy.

---

## 2.3 KBinsDiscretizer

### 1. Overview
Discretization converts a continuous feature into bins. It helps linear models represent non-linear threshold effects.

### 2. Concept & Formula
Strategies:
- **Uniform**: Equal-width intervals.
- **Quantile**: Equal number of observations per bin.
- **K-means**: 1D clustering.

### 3. Simple Explanation
Instead of feeding exact Age (23, 27, 45, 50) to a model, you put them in buckets: "20-30", "30-40", etc. This is useful if the risk profile changes in discrete steps rather than smoothly.

### 4. Manual Calculation
**Toy Incomes**: $x = (20, 25, 30, 60, 90)$. We want 3 bins using 'Uniform' strategy.
Range = $90 - 20 = 70$. Bin width = $70 / 3 = 23.33$.
- Bin 0: $[20, 43.33)$ $\rightarrow$ captures 20, 25, 30.
- Bin 1: $[43.33, 66.67)$ $\rightarrow$ captures 60.
- Bin 2: $[66.67, 90]$ $\rightarrow$ captures 90.

Output indices: $(0, 0, 0, 1, 2)$.

### 5. Code Python


In [ ]:
from sklearn.preprocessing import KBinsDiscretizer

incomes = np.array([20, 25, 30, 60, 90]).reshape(-1, 1)

kbd = KBinsDiscretizer(n_bins=3, encode='ordinal', strategy='uniform')
binned = kbd.fit_transform(incomes)

print("Bin Edges:", kbd.bin_edges_[0])
print("Binned Data:", binned.ravel())


### 6. Finance Perspective
Credit scoring models (like FICO) heavily use binning. A credit score of 720 vs 725 might not change default probability linearly, but jumping from the "Fair" bucket to the "Good" bucket has a massive step-effect. Binning continuous variables allows simple logistic regression models to capture these step-functions.

---

## 2.4 PolynomialFeatures

### 1. Overview
Creates powers and interactions ($a^2, ab, b^2$), allowing linear models to express nonlinear relationships.

### 2. Concept & Formula
For inputs $(a, b)$ and degree 2:
Features become: $[1, a, b, a^2, ab, b^2]$ (if bias is included).

### 3. Simple Explanation
A linear model can only draw a straight line. By giving it squared features, it can draw curves. By giving it interaction features ($a \times b$), it can understand that the combination of "High Income" AND "Low Debt" is exponentially better than just adding their individual effects.

### 4. Manual Calculation
**Input**: rows $(2, 3)$ and $(4, 5)$. Degree 2, no bias.
- Row 1 $(a=2, b=3) \mapsto (a, b, a^2, ab, b^2) = (2, 3, 4, 6, 9)$.
- Row 2 $(a=4, b=5) \mapsto (a, b, a^2, ab, b^2) = (4, 5, 16, 20, 25)$.

### 5. Code Python


In [ ]:
from sklearn.preprocessing import PolynomialFeatures

X_simple = np.array([[2, 3], [4, 5]])
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X_simple)

print("Feature Names:", poly.get_feature_names_out(['a', 'b']))
print("Transformed Data:\n", X_poly)


### 6. Finance Perspective
Interactions are vital in macro-trading. The effect of rising interest rates is completely different depending on whether inflation is high or low. A polynomial interaction term (`Rates * Inflation`) allows a linear model to capture this regime-dependent behavior.

---

## 2.5 Transforming the Target Variable ($y$)

### 1. Overview
Often, the target variable $y$ (e.g., house prices) is highly right-skewed. Training a regression model directly on $y$ can lead to poor results because the squared-error loss gets dominated by massive outlier values.

### 2. Concept & Formula
We train the model on $f(y)$ (e.g., $\log(1+y)$), and then predict using the inverse $f^{-1}(\hat{y})$ (e.g., $\exp(\hat{y}) - 1$).
Scikit-learn provides `TransformedTargetRegressor` to safely automate this.

### 3. Simple Explanation
If you try to predict house prices, a $\$10 \text{ million}$ mansion will throw off your model. So you predict the *logarithm* of the price instead. But your boss doesn't understand "the log price is 15". You must remember to exponentiate your prediction back to dollars before presenting it.

### 4. Manual Calculation
**House Prices**: $y = (100, 150, 400)$ (in USD thousands).
1. Transform: $\log(100) \approx 4.605$, $\log(150) \approx 5.011$, $\log(400) \approx 5.991$.
2. Model trains on these log values.
3. Model predicts a new house has $\hat{y}_{log} = 5.30$.
4. Invert prediction: $\exp(5.30) \approx 200.3$ USD thousands.

### 5. Code Python


In [ ]:
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

X = np.array([[1], [2], [3], [4]])
# Exponentially growing target
y = np.array([100, 150, 400, 1100]) 

# Using log1p (log(1+x)) and expm1 (exp(x)-1)
ttr = TransformedTargetRegressor(
    regressor=LinearRegression(),
    func=np.log1p,
    inverse_func=np.expm1
)

ttr.fit(X, y)
y_pred = ttr.predict([[2.5]])
print(f"Prediction for X=2.5 (in original units): {y_pred[0]:.2f}")


### 6. Finance Perspective
Target transformation is mandatory when forecasting asset prices or volatility. Financial series grow exponentially, not linearly. Forecasting the log-returns (or log-price) ensures that errors scale proportionally, preventing the model from over-optimizing for the absolute magnitude of recent, larger prices.

---

# Part 6: Practice Labs
---

## Lab 1: Comprehensive Pipeline on California Housing
**Tasks**:
1. Load the `fetch_california_housing` dataset.
2. Build a `ColumnTransformer` that:
   - Applies `RobustScaler` to skewed features (like Population).
   - Applies `StandardScaler` to normal features.
3. Use a `TransformedTargetRegressor` with a log transform for the target.
4. Fit and evaluate the model using Mean Absolute Error.



In [ ]:
# ==========================================================
# WRITE YOUR CODE HERE
# ==========================================================
pass

